# Joint Reconstruction and Classification for ECG Biomarkers

This notebook demonstrates the upgraded ECG Biomarker Encoder pipeline. The pipeline handles missing values using imputation and binary missingness masks, concatenates them into a joint feature vector, and trains models capable of performing **both** reconstruction (for latent clustering) and direct multi-label diagnostic classification.

### 1. Setup and Imports

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# Import custom pipeline components
from biomarker_encoder.preprocessing import BiomarkerPreprocessor
from biomarker_encoder.models import AttentionMLPAutoencoder, BetaVAE, FTTransformerAutoencoder
from biomarker_encoder.trainer import BiomarkerTrainer
from biomarker_encoder.evaluator import BiomarkerEvaluator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### 2. Load and Preprocess Data
The preprocessor automatically generates binary masks ($1.0$ if the biomarker exists, $0.0$ if missing), imputes missing values using the median, and scales the inputs.

In [ ]:
csv_path = "data/processed/full_biomarker_features.csv"
if not os.path.exists(csv_path):
    csv_path = "previous_version/biomarker_encoder/ecg_features.csv"

print(f"Loading data from: {csv_path}")
preprocessor = BiomarkerPreprocessor(random_state=42)
X_combined, y, df = preprocessor.load_and_preprocess(csv_path)

num_samples = X_combined.shape[0]
input_dim = X_combined.shape[1]
num_features = input_dim // 2

print(f"Total Samples: {num_samples}")
print(f"Input Dimension (Features + Mask): {input_dim} (Original Features: {num_features})")
print(f"Target Diagnostic Labels Shape: {y.shape}")

### 3. Create Splits and Dataloaders

In [ ]:
patient_ids = df["patient_id"].values if "patient_id" in df.columns else None
X_train, X_val, X_test, y_train, y_val, y_test = preprocessor.get_splits(
    X_combined, y, patient_ids=patient_ids
)

train_loader, val_loader, test_loader = preprocessor.get_dataloaders(
    X_train, X_val, X_test, y_train, y_val, y_test, batch_size=16
)

print(f"Train size: {len(X_train)} | Val size: {len(X_val)} | Test size: {len(X_test)}")

### 4. Build and Train Model
Let's train the **Attention MLP Autoencoder** as a manual demonstration.

In [ ]:
latent_dim = 32
model = AttentionMLPAutoencoder(
    input_dim=input_dim,
    latent_dim=latent_dim,
    dropout=0.2,
    num_heads=4,
    hidden_units=128,
    num_classes=5
)

trainer = BiomarkerTrainer(
    model=model,
    device=device,
    lr=1e-3,
    weight_decay=1e-4,
    patience=10,
    checkpoint_path="models_checkpoints/demo_attention_mlp.pt",
    mixed_precision=False
)

print("Starting demo training...")
model, train_history, val_history = trainer.fit(train_loader, val_loader, epochs=15)

# Plot Loss curves
plt.figure(figsize=(8, 5))
plt.plot(train_history, label="Train Loss")
plt.plot(val_history, label="Val Loss")
plt.title("Joint Training Loss (Reconstruction + Classification)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

### 5. Run Evaluation
Evaluate the reconstruction accuracy and the classification performance on the test set.

In [ ]:
evaluator = BiomarkerEvaluator(device=device)
metrics, test_embeddings, test_reconstructed, test_inputs = evaluator.evaluate_model(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    y_train=y_train,
    y_test=y_test
)

print("\n--- Evaluation Results ---")
print(f"Reconstruction MSE:           {metrics['MSE']:.6f}")
print(f"Reconstruction MAE:           {metrics['MAE']:.6f}")
print(f"Latent Silhouette Score:      {metrics['Silhouette_Score']:.4f}")
print(f"Direct Classifier F1-Score:   {metrics['Direct_F1_Score']:.4f}")
print(f"Direct Classifier ROC-AUC:    {metrics['Direct_ROC_AUC']:.4f}")
print(f"Downstream Linear Probe F1:   {metrics['Downstream_F1_Score']:.4f}")

### 6. Visualize Latent Space Clustering
Using t-SNE to project the 32-dim latent space into 2D.

In [ ]:
perplexity = min(30, max(1, len(test_embeddings) - 1))
tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42)
tsne_coords = tsne.fit_transform(test_embeddings)

# Color by dominant diagnostic class
class_labels = np.argmax(y_test, axis=1)
label_names = preprocessor.label_cols

plt.figure(figsize=(8, 6))
scatter = plt.scatter(tsne_coords[:, 0], tsne_coords[:, 1], c=class_labels, cmap='viridis', alpha=0.8)
plt.colorbar(scatter, ticks=range(len(label_names)), format=plt.FuncFormatter(lambda val, loc: label_names[int(val)]))
plt.title("t-SNE Visualization of Biomarker Latent Embeddings")
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.grid(True)
plt.show()